# 04 — Advanced Analysis: Clustering & PCA

Identify country archetypes based on emissions profiles using KMeans clustering and PCA. Explore how income, fuel mix, and energy efficiency define distinct climate pathways.

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT  = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR       = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR    = PROJECT_ROOT / "reports" / "figures"
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

section("Load modern country-level dataset")
modern = pd.read_csv(PROCESSED_DIR / "co2_modern.csv")
print(f"Shape: {modern.shape}")
display(modern.head(3))



Load modern country-level dataset
Shape: (7630, 87)


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,co2_including_luc_growth_abs,co2_including_luc_growth_prct,co2_including_luc_per_capita,co2_including_luc_per_gdp,co2_including_luc_per_unit_energy,co2_per_capita,co2_per_gdp,co2_per_unit_energy,coal_co2,coal_co2_per_capita,consumption_co2,consumption_co2_per_capita,consumption_co2_per_gdp,cumulative_cement_co2,cumulative_co2,cumulative_co2_including_luc,cumulative_coal_co2,cumulative_flaring_co2,cumulative_gas_co2,cumulative_luc_co2,cumulative_oil_co2,cumulative_other_co2,energy_per_capita,energy_per_gdp,flaring_co2,flaring_co2_per_capita,gas_co2,gas_co2_per_capita,ghg_excluding_lucf_per_capita,ghg_per_capita,land_use_change_co2,land_use_change_co2_per_capita,methane,methane_per_capita,nitrous_oxide,nitrous_oxide_per_capita,oil_co2,oil_co2_per_capita,other_co2_per_capita,other_industry_co2,primary_energy_consumption,share_global_cement_co2,share_global_co2,share_global_co2_including_luc,share_global_coal_co2,share_global_cumulative_cement_co2,share_global_cumulative_co2,share_global_cumulative_co2_including_luc,share_global_cumulative_coal_co2,share_global_cumulative_flaring_co2,share_global_cumulative_gas_co2,share_global_cumulative_luc_co2,share_global_cumulative_oil_co2,share_global_cumulative_other_co2,share_global_flaring_co2,share_global_gas_co2,share_global_luc_co2,share_global_oil_co2,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share,co2_per_energy,fossil_share,luc_share,gdp_per_capita,co2_growth_cat,income_tier,dominant_fuel,decade
0,Afghanistan,1990,AFG,12045664.0,1.306598e+10,0.045766,0.003799,2.024326,-0.740529,-26.783646,2.196790,-1.491686,-40.441780,0.182372,0.168131,72.748940,0.168054,0.154931,67.037605,0.278464,0.023117,NaN,NaN,NaN,1.536562,58.603493,343.539398,11.510028,5.710451,10.813995,778.365112,29.032457,NaN,2506.866699,2.311106,0.025648,0.002129,0.403040,0.033459,0.259268,1.122328,0.172464,0.014318,6.733515,0.558999,2.675560,0.222118,1.271408,0.105549,NaN,NaN,30.196873,0.009292,0.008905,0.007692,0.003216,0.012341,0.007261,0.022795,0.002631,0.069859,0.013741,0.110472,0.010818,NaN,0.009988,0.010634,0.002959,0.013825,NaN,0.089654,0.000306,0.000493,0.000881,0.000083,13.519183,3.123060,NaN,NaN,0.067038,0.964722,0.078507,1084.704338,Sharp decline (>-5%),Low income,Oil,1990s
1,Afghanistan,1991,AFG,12238879.0,1.204736e+10,0.045766,0.003739,1.914301,-0.110025,-5.435151,1.939509,-0.257281,-11.711681,0.158471,0.160990,155.200272,0.156411,0.158898,153.183105,0.249627,0.020396,NaN,NaN,NaN,1.582328,60.517792,345.478912,11.759655,5.736148,11.203120,778.390320,30.236544,NaN,1021.075195,1.037307,0.025697,0.002100,0.389125,0.031794,0.248190,1.120257,0.025208,0.002060,7.021835,0.573732,2.749918,0.224687,1.204085,0.098382,NaN,NaN,12.496816,0.009068,0.008249,0.006705,0.002911,0.012214,0.007289,0.022492,0.002637,0.067912,0.013572,0.109586,0.010876,NaN,0.009442,0.010103,0.000441,0.012469,NaN,0.087987,0.000302,0.000495,0.000880,0.000084,13.710694,3.037569,NaN,NaN,0.153183,0.962668,0.012997,984.351741,Sharp decline (>-5%),Low income,Oil,1990s
2,Afghanistan,1992,AFG,13278983.0,1.267754e+10,0.045766,0.003446,1.482054,-0.432247,-22.579884,-0.178617,-2.118127,-109.209404,-0.013451,-0.014089,-21.763641,0.111609,0.116904,180.580933,0.021984,0.001656,NaN,NaN,NaN,1.628094,61.999847,345.300293,11.781639,5.758132,11.565856,776.729675,31.266129,NaN,618.055298,0.647377,0.021984,0.001656,0.362736,0.027317,0.196275,0.869253,-1.660671,-0.125060,7.107602,0.535252,2.688141,0.202436,1.029584,0.077535,NaN,NaN,8.207146,0.008702,0.006581,-0.000627,0.000263,0.012077,0.007270,0.022071,0.002593,0.066290,0.013379,0.108441,0.010888,NaN,0.009168,0.009298,-0.027827,0.011247,NaN,0.086610,0.000300,0.000495,0.000881,0.000085,11.542790,2.606333,NaN,NaN,0.180

In [2]:
section("Build cross-sectional snapshot for clustering (latest available year per country)")

# Use the most recent year that has co2 data for each country
snap = (
    modern
    .dropna(subset=["co2","co2_per_capita"])
    .sort_values("year")
    .groupby("country")
    .last()
    .reset_index()
)
print(f"Snapshot countries: {len(snap)}")

# Clustering features — choose columns with reasonable coverage
CLUSTER_FEATURES = [
    "co2_per_capita",
    "co2_per_gdp",
    "fossil_share",
    "energy_per_capita",
    "gdp_per_capita",
    "co2_growth_prct",
    "share_global_co2",
]
CLUSTER_FEATURES = [c for c in CLUSTER_FEATURES if c in snap.columns]
print(f"Clustering features: {CLUSTER_FEATURES}")

# Keep only rows with all features
snap_clean = snap.dropna(subset=CLUSTER_FEATURES).copy()
print(f"Countries with complete features: {len(snap_clean)}")
display(snap_clean[["country"] + CLUSTER_FEATURES].head())



Build cross-sectional snapshot for clustering (latest available year per country)
Snapshot countries: 213
Clustering features: ['co2_per_capita', 'co2_per_gdp', 'fossil_share', 'energy_per_capita', 'gdp_per_capita', 'co2_growth_prct', 'share_global_co2']
Countries with complete features: 164


,country,co2_per_capita,co2_per_gdp,fossil_share,energy_per_capita,gdp_per_capita,co2_growth_prct,share_global_co2
0,Afghanistan,0.253848,0.190792,0.999019,925.085266,1313.577777,2.944744,0.028048
1,Albania,1.591990,0.124362,0.795850,9167.564453,12792.059169,0.613832,0.011515
2,Algeria,4.233817,0.323552,0.860473,16237.188477,13101.456658,-2.289152,0.513499
4,Angola,0.589497,0.133183,0.776202,2659.693115,4443.555847,3.039324,0.057861
7,Argentina,3.743406,0.214955,0.963355,21513.947266,18827.435063,-4.118520,0.443175


In [3]:
section("Scale features & evaluate cluster count")

X = snap_clean[CLUSTER_FEATURES].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"X_scaled shape: {X_scaled.shape}")

eval_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    eval_rows.append({
        "k": k,
        "Inertia": km.inertia_,
        "Silhouette": silhouette_score(X_scaled, labels)
    })
eval_df = pd.DataFrame(eval_rows)
display(eval_df.round(4))

fig1 = px.line(eval_df, x="k", y="Inertia", markers=True, title="Elbow Method — KMeans Inertia")
fig1.show()
fig2 = px.line(eval_df, x="k", y="Silhouette", markers=True, title="Silhouette Score by k")
fig2.show()
print("\nWe use k=4 — balances interpretability with silhouette quality.")



Scale features & evaluate cluster count
X_scaled shape: (164, 7)


,k,Inertia,Silhouette
0,2,859.5488,0.4858
1,3,720.1280,0.2588
2,4,596.3784,0.2675
3,5,501.9366,0.2699
4,6,415.1266,0.2935
5,7,373.3770,0.2855
6,8,344.3358,0.2653



We use k=4 — balances interpretability with silhouette quality.


In [4]:
section("Fit KMeans (k=4) and name clusters")

K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=20)
snap_clean["Cluster"] = kmeans.fit_predict(X_scaled)

# Profile each cluster
profile = snap_clean.groupby("Cluster")[CLUSTER_FEATURES].mean().round(3)
display(profile)

# Name clusters based on their dominant characteristics
cluster_names = {}
for cid, row in profile.iterrows():
    high_fossil  = row.get("fossil_share", 0) > profile["fossil_share"].median()
    high_gdp     = row.get("gdp_per_capita", 0) > profile["gdp_per_capita"].median()
    high_co2cap  = row.get("co2_per_capita", 0) > profile["co2_per_capita"].median()
    high_growth  = row.get("co2_growth_prct", 0) > 0

    if high_gdp and high_co2cap and high_fossil:
        name = "Fossil-intensive Rich"
    elif high_gdp and not high_fossil:
        name = "Clean-transitioning Rich"
    elif not high_gdp and high_growth:
        name = "Rapidly industrialising"
    else:
        name = "Low-emission Developing"
    cluster_names[cid] = name

# Deduplicate names
seen, final_names = {}, {}
for cid, name in cluster_names.items():
    seen[name] = seen.get(name, 0) + 1
    final_names[cid] = name if seen[name] == 1 else f"{name} ({seen[name]})"

snap_clean["Cluster_Name"] = snap_clean["Cluster"].map(final_names)
print("\nCluster assignments:")
display(snap_clean["Cluster_Name"].value_counts().to_frame("Countries"))

# Show representative countries per cluster
for cname in snap_clean["Cluster_Name"].unique():
    sub = snap_clean[snap_clean["Cluster_Name"]==cname]
    print(f"\n{cname}: {sub.nlargest(8,'co2')['country'].tolist()}")



Fit KMeans (k=4) and name clusters


,co2_per_capita,co2_per_gdp,fossil_share,energy_per_capita,gdp_per_capita,co2_growth_prct,share_global_co2
Cluster,,,,,,,
0,6.033,0.290,0.949,34928.241,30049.976,-1.643,0.442
1,1.570,0.206,0.905,7330.303,7493.245,2.747,0.232
2,8.658,0.434,0.934,34514.457,18921.136,0.961,31.838
3,18.350,0.369,0.961,121204.413,59953.945,2.097,1.444



Cluster assignments:


,Countries
Cluster_Name,
Rapidly industrialising,96
Low-emission Developing,54
Fossil-intensive Rich,13
Rapidly industrialising (2),1



Rapidly industrialising: ['India', 'Indonesia', 'Turkey', 'Brazil', 'Mexico', 'Vietnam', 'Thailand', 'Egypt']

Low-emission Developing: ['Russia', 'Japan', 'Iran', 'South Korea', 'Germany', 'South Africa', 'United Kingdom', 'Italy']

Fossil-intensive Rich: ['United States', 'Saudi Arabia', 'Canada', 'Australia', 'United Arab Emirates', 'Kuwait', 'Qatar', 'Oman']

Rapidly industrialising (2): ['China']


In [5]:
section("Cluster profile comparison")

profile_long = (
    snap_clean.groupby("Cluster_Name")[CLUSTER_FEATURES].mean()
    .reset_index()
    .melt(id_vars="Cluster_Name", var_name="Metric", value_name="Average")
)

fig = px.bar(
    profile_long, x="Metric", y="Average", color="Cluster_Name", barmode="group",
    title="Country Cluster Profile Comparison (Unscaled Averages)",
    labels={"Metric": "", "Average": "Mean value"}
)
fig.update_layout(xaxis_tickangle=-30, height=560)
fig.show()

# Radar chart per cluster
from plotly.subplots import make_subplots

# Normalise to 0-1 for radar
norm = snap_clean.groupby("Cluster_Name")[CLUSTER_FEATURES].mean()
norm_scaled = (norm - norm.min()) / (norm.max() - norm.min())

fig_radar = go.Figure()
for cname in norm_scaled.index:
    values = norm_scaled.loc[cname].tolist()
    values += [values[0]]  # close the loop
    fig_radar.add_trace(go.Scatterpolar(
        r=values, theta=CLUSTER_FEATURES + [CLUSTER_FEATURES[0]],
        fill="toself", name=cname
    ))
fig_radar.update_layout(
    title="Cluster Radar Chart (Normalised 0–1)",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=550
)
fig_radar.show()



Cluster profile comparison


In [6]:
section("PCA — 2D behavioural space")

pca = PCA(n_components=2, random_state=42)
pts = pca.fit_transform(X_scaled)
snap_clean["PCA_1"] = pts[:, 0]
snap_clean["PCA_2"] = pts[:, 1]

print(f"PCA component 1 explains: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"PCA component 2 explains: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# PCA loadings
loadings = pd.DataFrame(
    pca.components_.T, index=CLUSTER_FEATURES, columns=["PC1","PC2"]
).round(3)
print("\nPCA loadings (feature contributions):")
display(loadings)

fig = px.scatter(
    snap_clean, x="PCA_1", y="PCA_2", color="Cluster_Name",
    text="country", opacity=0.8,
    hover_data=["co2_per_capita","gdp_per_capita","fossil_share","energy_per_capita"],
    title="PCA Country Map — Emission Profile Space<br>"
          f"<sub>PC1 {pca.explained_variance_ratio_[0]*100:.1f}% | PC2 {pca.explained_variance_ratio_[1]*100:.1f}%</sub>",
    labels={"PCA_1": "PC1", "PCA_2": "PC2"}
)
fig.update_traces(textposition="top center", textfont_size=8)
fig.update_layout(height=700)
fig.show()



PCA — 2D behavioural space
PCA component 1 explains: 40.6%
PCA component 2 explains: 16.3%
Total explained variance: 56.9%

PCA loadings (feature contributions):


,PC1,PC2
co2_per_capita,0.555,0.091
co2_per_gdp,0.185,0.765
fossil_share,0.224,-0.171
energy_per_capita,0.560,-0.129
gdp_per_capita,0.520,-0.308
co2_growth_prct,-0.100,-0.384
share_global_co2,0.117,0.344


In [7]:
section("Density contour — CO₂ per capita vs GDP per capita")

plot_data = snap_clean.dropna(subset=["gdp_per_capita","co2_per_capita"])

fig = px.density_contour(
    plot_data, x="gdp_per_capita", y="co2_per_capita",
    color="Cluster_Name",
    title="Density Contour — GDP per Capita vs CO₂ per Capita by Cluster",
    labels={"gdp_per_capita": "GDP per capita (USD)", "co2_per_capita": "CO₂ per capita (t)"}
)
fig.update_traces(contours_coloring="fill", contours_showlabels=True)
fig.update_layout(height=550)
fig.show()



Density contour — CO₂ per capita vs GDP per capita


In [8]:
section("Parallel coordinates — high-dimensional emissions fingerprint")

sample = snap_clean.sample(min(200, len(snap_clean)), random_state=42).copy()
codes  = {n: i for i, n in enumerate(sorted(snap_clean["Cluster_Name"].unique()))}
sample["Cluster_Code"] = sample["Cluster_Name"].map(codes)

para_cols = [c for c in ["co2_per_capita","fossil_share","energy_per_capita",
                         "gdp_per_capita","co2_per_gdp","co2_growth_prct"]
             if c in sample.columns]

fig = px.parallel_coordinates(
    sample, dimensions=para_cols, color="Cluster_Code",
    color_continuous_scale=px.colors.diverging.Tealrose,
    title="Parallel Coordinates — Country Emissions Fingerprint by Cluster"
)
fig.update_layout(height=520)
fig.show()



Parallel coordinates — high-dimensional emissions fingerprint


In [9]:
section("Temporal cluster trajectory — how clusters evolve over time")

# Attach cluster labels back to the full modern dataset
label_map = snap_clean.set_index("country")["Cluster_Name"].to_dict()
modern["Cluster_Name"] = modern["country"].map(label_map)

trend = (
    modern.dropna(subset=["Cluster_Name","co2_per_capita"])
    .groupby(["year","Cluster_Name"], as_index=False)
    ["co2_per_capita"].mean()
)

fig = px.line(
    trend, x="year", y="co2_per_capita", color="Cluster_Name",
    title="CO₂ per Capita Trend by Cluster (1990–present)",
    labels={"co2_per_capita": "Mean CO₂ per capita (t)", "year": "Year"}
)
fig.update_layout(height=480)
fig.show()



Temporal cluster trajectory — how clusters evolve over time


In [10]:
section("Export clustered datasets")

snap_clean.to_csv(PROCESSED_DIR / "co2_clustered.csv", index=False)
snap_clean.groupby("Cluster_Name")[CLUSTER_FEATURES].mean().round(3).reset_index().to_csv(
    PROCESSED_DIR / "cluster_profiles.csv", index=False
)
size = snap_clean["Cluster_Name"].value_counts().reset_index()
size.columns = ["Cluster_Name","Countries"]
size["Share"] = (size["Countries"] / len(snap_clean)).round(3)
size.to_csv(PROCESSED_DIR / "cluster_sizes.csv", index=False)
print("Exported: co2_clustered.csv, cluster_profiles.csv, cluster_sizes.csv")
display(size)



Export clustered datasets
Exported: co2_clustered.csv, cluster_profiles.csv, cluster_sizes.csv


,Cluster_Name,Countries,Share
0,Rapidly industrialising,96,0.585
1,Low-emission Developing,54,0.329
2,Fossil-intensive Rich,13,0.079
3,Rapidly industrialising (2),1,0.006
